In [2]:
import pandas as pd
import re

import torch

In [3]:
#extracting datasets
df_cnbc = pd.read_csv('Dataset/cnbc_headlines.csv')
df_guardian = pd.read_csv('Dataset/guardian_headlines.csv')
df_reuters = pd.read_csv('Dataset/reuters_headlines.csv')

#considering only headlines
headline_cnbc = list(df_cnbc['Headlines'])
headline_guardian = list(df_guardian['Headlines'])
headline_reuters = list(df_reuters['Headlines'])

#list of headlines
headline_list = headline_cnbc + headline_guardian + headline_reuters

print("No.of headlines in list: ",len(headline_list))
print(headline_list[:5])


No.of headlines in list:  53650
['Jim Cramer: A better way to invest in the Covid-19 vaccine gold rush', "Cramer's lightning round: I would own Teradyne", nan, "Cramer's week ahead: Big week for earnings, even bigger week for vaccines", 'IQ Capital CEO Keith Bliss says tech and healthcare will rally']


In [4]:
#cleaning dataset

#filtering empty rows
headline_list = [h for h in headline_list if str(h) != 'nan']
print("No.of headlines in list: ",len(headline_list))
print(headline_list[:5])

#removing spaces in headline
def clean_headline(h):
    h = h.strip() #remove the trailing spaces from either end
    h = re.sub(r'\s+', ' ', h) #replace bigger spaces with a single space
    h = h.encode('ascii', 'ignore').decode()  # remove non-ASCII
    return h

headline_list = [clean_headline(h) for h in headline_list]



No.of headlines in list:  53370
['Jim Cramer: A better way to invest in the Covid-19 vaccine gold rush', "Cramer's lightning round: I would own Teradyne", "Cramer's week ahead: Big week for earnings, even bigger week for vaccines", 'IQ Capital CEO Keith Bliss says tech and healthcare will rally', "Wall Street delivered the 'kind of pullback I've been waiting for,' Jim Cramer says"]


In [5]:
#adding start and end tags before each letter
word_dict = {}
for h in headline_list:
    chs = ['<S>'] + list(h) + ['<E>']
    for ch1, ch2 in zip(chs, chs[1:]):
        bigram = (ch1, ch2)
        word_dict[bigram] = word_dict.get(bigram, 0) + 1 
        #returns 0 if none present, else corresponding dictionary bigram value

sorted(word_dict.items(), key = lambda kv : -kv[1])


[(('s', ' '), 97766),
 (('i', 'n'), 57628),
 (('e', ' '), 55140),
 (('e', 'r'), 46323),
 ((' ', 't'), 45466),
 (('n', ' '), 43826),
 (('e', 's'), 42548),
 ((' ', 's'), 41496),
 (('t', ' '), 41297),
 (('o', 'n'), 40240),
 (('r', 'e'), 36807),
 (('a', 'n'), 35004),
 ((' ', 'a'), 34130),
 (('r', ' '), 33127),
 (('o', 'r'), 32311),
 (('t', 'o'), 31405),
 (('s', 't'), 29152),
 ((' ', 'c'), 28882),
 (('a', 'r'), 28454),
 ((' ', 'o'), 27551),
 (('d', ' '), 26919),
 ((' ', 'f'), 26815),
 (('a', 'l'), 25530),
 (('o', ' '), 25437),
 ((' ', 'b'), 23939),
 ((' ', 'i'), 23398),
 (('t', 'e'), 23058),
 (('i', 't'), 22907),
 (('e', 'n'), 22518),
 (('n', 'g'), 22480),
 ((' ', 'p'), 21660),
 (('y', ' '), 21642),
 (('a', 't'), 20146),
 (('r', 'o'), 19930),
 (('d', 'e'), 19842),
 (('r', 'a'), 19800),
 (('v', 'e'), 19760),
 ((' ', 'r'), 19573),
 (('t', 'i'), 19244),
 (('l', ' '), 19120),
 (('t', 'h'), 19080),
 (('r', 'i'), 19013),
 (('s', 'e'), 18875),
 (('n', 'd'), 18767),
 (('e', 'a'), 18645),
 (('c', 'o

In [6]:
unique_words = set()

for headline in headline_list:
    for letter in list(headline):
        unique_words.add(letter)
    
unique_words = ['<S>', '<E>'] + sorted(unique_words)
print("Vocabulary size: ",len(unique_words))

Vocabulary size:  84


In [7]:
#number to word matrix
num2letter_dict = {i:letter for i, letter in enumerate(unique_words)}

#word to number matrix
letter2num_dict = {letter:i for i, letter in enumerate(unique_words)}


In [8]:
#creating the count matrix
N = torch.zeros((len(unique_words), len(unique_words)), dtype = torch.int32)

for h in headline_list:
    chs = ['<S>'] + list(h) + ['<E>']
    for ch1, ch2 in zip(chs, chs[1:]):
        N[letter2num_dict[ch1]][letter2num_dict[ch2]] += 1 #increment corresponding set of bigrams by 1
    

#now, calculating probabilities out of the counts
P = (N+1).float() / (N+1).float().sum(dim=1, keepdim=True)
print("Probability: ", P[letter2num_dict['s']])

Probability:  tensor([4.1925e-06, 6.4115e-02, 4.0988e-01, 5.8695e-05, 8.3849e-06, 4.1925e-06,
        4.1925e-06, 4.1925e-06, 4.1925e-06, 5.5508e-03, 4.1925e-06, 1.6770e-05,
        4.1925e-06, 8.3849e-06, 1.4263e-02, 1.6099e-03, 4.2763e-04, 7.5464e-05,
        4.1925e-06, 4.1925e-06, 4.1925e-06, 4.1925e-06, 8.3849e-06, 1.2577e-05,
        4.1925e-06, 4.1925e-06, 4.1925e-06, 4.1925e-06, 8.6239e-03, 1.2410e-03,
        1.4003e-03, 8.8042e-05, 4.1925e-06, 2.0962e-05, 4.1925e-06, 4.1925e-06,
        8.3849e-06, 4.1925e-06, 4.1925e-06, 4.1925e-06, 4.1925e-06, 4.1925e-06,
        4.1925e-06, 8.3849e-06, 4.1925e-06, 4.1925e-06, 1.6770e-05, 4.1925e-06,
        4.1925e-06, 4.1925e-06, 4.1925e-06, 8.3849e-06, 4.1925e-06, 8.3849e-06,
        8.3849e-06, 4.1925e-06, 8.3849e-06, 4.1925e-06, 4.0030e-02, 1.0565e-03,
        1.2359e-02, 1.7021e-03, 7.9137e-02, 7.5045e-04, 1.9285e-04, 3.9032e-02,
        5.3031e-02, 8.3849e-06, 7.5926e-03, 1.1860e-02, 4.9765e-03, 4.0960e-03,
        2.3172e-02, 1.6170

In [9]:
def generate_headline():
    current = '<S>'
    headline = []
    while True:
        get_index = letter2num_dict[current]
        prob_row = P[get_index] #row of probabilities of the respective character
        
        #now, considering prob_row as weights for multinomial
        char_index = torch.multinomial(prob_row, num_samples=1).item()
        current = num2letter_dict[char_index] #picks the char of the respective index

        #check for end char and break
        if current == '<E>':
            break
        headline.append(current)

    return ''.join(headline)

for _ in range(5):
    print(generate_headline())

#to intuitively think of torch.multinomial, think of a spinning wheel, where the higher probability gets a bigger chunk
#hence, it is like spinning that wheel, where it randomly selects the next occurence, the chances higher/lower based on the given probability of the character


Tors Chimingit f g tan res HSprency, sty BSk25m wsscatooly Pesterinit toity l id theans tecocod wimp cisutadusur dsctes lins t cenckeey U.9/S.Sistievencke Allecaduracoloms coce ne
UVeks U. as o clyd frerteadis tr bawhiger onev Thire UK tr ctean fot s tast l arderinaspur stred tnes
Cofft'store'slewesias mis It U.Stinddecourutofthoh ftidus Goise ul f powstomallnk one ratores, poboct Jend?
Jumuned-e
Vot gon As t fad the awamewt oumorad foditlepollall Ge fop Fowesopoumeroushw twis-ce ftin UK nbuppa lllyst tes at-Ferort aghtairtoons Raliciper ps: s Honorilaxeix-bay beve juniamaft uckanomac ces gueclonetont decorlone m UK, dompertode eraxpl trlive tls
